In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
3 * 36 * 36

In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.optim import AdamW
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test  = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test  = torch.tensor(y_test, dtype=torch.float32)

# convert data to tensor for apel to trine it in pytorch "NN"

In [ ]:
# 2. Create TensorDataset objects
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

#combine x and y to make dataloder later


In [ ]:
# 3. Create DataLoaders

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
# now the data is ready :)



In [ ]:
# 4. Print shape of one batch
X_batch, y_batch = next(iter(train_loader))
print(f"Training batch input shape: {X_batch.shape}")
print(f"Training batch labels shape: {y_batch.shape}")


In [ ]:
images, labels = next(iter(train_loader))

# Display the first 6 images in the batch
plt.figure(figsize=(8, 4))

for i in range(6):
    plt.subplot(2, 3, i + 1)

    img = images[i].permute(1, 2, 0)

    plt.imshow(img)
    plt.title(f"Label: {labels[i].item()}")
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Task 1: Write your model class here:
class NN4Layer(nn.Module):

    def __init__(self, input_dim, hidden_dim, output_dim):
        super(NN4Layer, self).__init__()

        self.layer1 = nn.Linear(input_dim, hidden_dim)
        self.layer2 = nn.Linear(hidden_dim, hidden_dim // 2)
        self.layer3 = nn.Linear(hidden_dim // 2, hidden_dim // 4)
        self.layer4 = nn.Linear(hidden_dim // 4, output_dim)
# i cut half if hidden_dim every time idont konw why but it looks more cool


        self.batchnorm1 = nn.BatchNorm1d(hidden_dim)
        self.batchnorm2 = nn.BatchNorm1d(hidden_dim // 2)
        self.batchnorm3 = nn.BatchNorm1d(hidden_dim // 4)
#BatchNorm1d help the model to get better acc
# التطبيع يقدر انه يحسن الموديل بانه يقرب الاعداد ويسوي شي يشابه السكالير لهذا السبب راج استعمله

        # activation function for non-linearity
        self.relu = nn.ReLU()

        self.dropout = nn.Dropout(0.2)
# الدروب اوت من اجل ان الموديل مايصير له اوفر فتنق ويبدا يهلوس



    def forward(self, x):
        # Layer 1:
        z1 = self.batchnorm1(self.layer1(x))
        a1 = self.dropout(self.relu(z1))

        # Layer 2
        z2 = self.batchnorm2(self.layer2(a1))
        a2 = self.dropout(self.relu(z2))

        # Layer 3
        z3 = self.batchnorm3(self.layer3(a2))
        a3 = self.dropout(self.relu(z3))

        # Output layer
        z5 = (self.layer4(a3))


        return z5

In [ ]:
# Task 2: Write your training loop here:
def train_one_epoch(model, optimizer, criterion, train_loader, device):
  model.train()
  #set the model to trine mode

  running_loss = 0.0

  for X_batch, y_batch in train_loader:
    X_batch = X_batch.view(X_batch.size(0), -1).to(device)
    y_batch = y_batch.to(device)
    # put the data into the device



    outputs = model(X_batch)
    #get model predictions

    loss = criterion(outputs, y_batch)
    #calcute the loss


    #Clear previous gradients
    optimizer.zero_grad()


    #Compute gradients (backward pass)
    loss.backward()

    #Update model parameters
    optimizer.step()

    running_loss += loss.item()

  # Calculate average loss over all batches
  avg_loss = running_loss / len(train_loader)

  return avg_loss

In [ ]:
# Task 3: Write your validation loop here:
def validate(model, criterion, test_loader, device):

  model.eval()
  running_loss = 0.0

  #with torch.no_grad() Disable gradient computation no need calc update W
  with torch.no_grad():
    for X_batch, y_batch in test_loader:

      X_batch = X_batch.view(X_batch.size(0), -1).to(device)
      y_batch = y_batch.to(device)

      # get model predictions
      outputs = model(X_batch)

      #calcute the loss
      loss = criterion(outputs, y_batch)

      running_loss += loss.item()

  avg_loss = running_loss / len(test_loader)

  return avg_loss

In [ ]:
# Task 4: Define device, model, loss, optimizer:

#check if cuda available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#define input dim

input_dim = 3 * 36 * 36

#define hidden shape
hidden_dim = 512

#define model
model = NN4Layer(input_dim, hidden_dim, 1)

#define loss function
criterion = nn.MSELoss()

#define optimizer im using adamW "some of the best optimizers"
optimizer = AdamW(model.parameters(),lr=0.001)

num_epochs = 20



In [ ]:
# Task 5: Start training for 20 epochs:

# save model loss history
train_losses = []
val_losses = []

num_epochs = 20

print('Starting Training...')
for epoch in range(num_epochs):

  train_loss = train_one_epoch(model, optimizer, criterion, train_loader, device)


  val_loss = validate(model, criterion, test_loader, device)

  train_losses.append(train_loss)
  val_losses.append(val_loss)

  print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}')

print('Training Complete!')

In [ ]:
# Task 1: Write your code here:
# Plotting results
plt.figure(figsize=(7, 5))

plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Task 2 (Bonus): Write your code here:
# Set model to evaluation mode
model.eval()
# Get one batch from the test DataLoader
images, labels = next(iter(test_loader))

# Move images to device
images_test = images.view(X_batch.size(0), -1).to(device)

with torch.no_grad():
    predictions = outputs = model(images_test)

# Move tensors back to CPU for plotting
images = images.cpu()
labels = labels.cpu()
predictions = predictions.cpu()

# Plot first 6 predictions
plt.figure(figsize=(15, 10))
for i in range(6):
    plt.subplot(2, 3, i + 1)
    img = images[i].permute(1, 2, 0)
    plt.imshow(img)
    plt.title(f"True: {labels[i]} | Pred: {predictions[i]}")
    plt.axis('off')

plt.tight_layout()
plt.show()